#1~4で最も高いスコアを記録したデータ・モデルを用いて、さらに改良を行う。

In [1]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [2]:
#データフレームの読み込み
'''
以降はスコアが最高であったdf3を用いて分析を行う。
sc_x,df_yはdf3を基に作成する。
ただし、必要があればdf1, df2も利用する。
'''
df3 = pd.read_csv('datafiles/df3_lowered_VIF.csv')

sc_x = df3.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df3['SalePrice'])

In [3]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
#リッジ回帰のalphaを90~1000で最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(90,1000):
    ridgeModel = Ridge(random_state = 0, alpha = i)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = i
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')

最適な正則化項＝478　最高スコア＝0.8205844141029276


In [ ]:
#リッジ回帰のalphaを478付近でより細かく最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(47700,47900):
    num = i/100
    ridgeModel = Ridge(random_state = 0, alpha = num)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')


最適な正則化項＝478.14　最高スコア＝0.8205844152693619


In [ ]:
#4_analysis_2においてVIFを行の削除により下げたスコアの最高値   0.7971109145093968よりもスコアが向上した。

In [14]:
#引き続き、特徴量の最適化を行う。
model5 = Ridge(random_state = 0, alpha = best_alpha)
model5.fit(sc_x, df_y)
coef_df = pd.DataFrame({
    'features':sc_x.columns,
    'Coefficient': model5.coef_
})
coef_df.sort_values('Coefficient', ascending=False)

,features,Coefficient
2,OverallQual,10957.831681
8,1stFlrSF,9102.571118
16,TotRmsAbvGrd,7851.661035
195,Neighborhood_NoRidge,6941.700492
196,Neighborhood_NridgHt,6872.611172
...,...,...
75,BldgType_TwnhsE,-3760.381477
127,KitchenQual_Gd,-4433.561103
48,BsmtQual_Gd,-4760.329032
97,PoolQC_Gd,-4799.248606


In [ ]:
'''
今回のデータセットでは各特徴量の意味が明確であるから、ドメイン知識に基づいて仮説検証を行う。
以下が検証すべき仮説である。
・


'''